# 12 - Teste Final

**Este e o teste final. `test.parquet` e carregado UMA vez, neste notebook, pela
primeira e unica vez em todo o projeto.**

A configuracao dos modelos esta CONGELADA - hiperparametros, thresholds e features foram
fixados no notebook 11, olhando apenas a validacao (2014). Nada aqui e reajustado com
base no que este notebook mostrar. Se o resultado for decepcionante, ele e reportado como
esta.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from xgboost import XGBClassifier
import warnings

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)

%matplotlib inline

PROCESSED_DIR = Path('..') / 'data' / 'processed'

train = pd.read_parquet(PROCESSED_DIR / 'train.parquet')
validation = pd.read_parquet(PROCESSED_DIR / 'validation.parquet')
test = pd.read_parquet(PROCESSED_DIR / 'test.parquet')
transfer_60m = pd.read_parquet(PROCESSED_DIR / 'transfer_60m.parquet')

print(f'train: {train.shape} | validation: {validation.shape} | test: {test.shape} | transfer_60m: {transfer_60m.shape}')


train: (172988, 89) | validation: (162570, 89) | test: (282787, 89) | transfer_60m: (54969, 89)


### Reconstrucao de FEATURE_SET, financeiro, prepare_X (identico aos notebooks 06-11)

In [2]:
EVAL_ONLY = ['loan_status', 'loan_amnt', 'installment', 'term', 'total_rec_prncp']
PROVISIONAL_EXCLUDE = ['int_rate', 'grade', 'sub_grade']

family_C_features = ['funded_amnt', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d',
    'purpose', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high',
    'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec',
    'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'collections_12_mths_ex_med',
    'mths_since_last_major_derog', 'application_type', 'acc_now_delinq', 'tot_coll_amt',
    'tot_cur_bal', 'total_rev_hi_lim', 'acc_open_past_24mths', 'avg_cur_bal', 'bc_open_to_buy',
    'bc_util', 'chargeoff_within_12_mths', 'delinq_amnt', 'mo_sin_old_il_acct',
    'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl', 'mort_acc',
    'mths_since_recent_bc', 'mths_since_recent_bc_dlq', 'mths_since_recent_inq',
    'mths_since_recent_revol_delinq', 'num_accts_ever_120_pd', 'num_actv_bc_tl', 'num_actv_rev_tl',
    'num_bc_sats', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl', 'num_rev_accts',
    'num_rev_tl_bal_gt_0', 'num_sats', 'num_tl_120dpd_2m', 'num_tl_30dpd', 'num_tl_90g_dpd_24m',
    'num_tl_op_past_12m', 'pct_tl_nvr_dlq', 'percent_bc_gt_75', 'pub_rec_bankruptcies',
    'tax_liens', 'tot_hi_cred_lim', 'total_bal_ex_mort', 'total_bc_limit',
    'total_il_high_credit_limit', 'emp_length_anos']
assert len(family_C_features) == 65

engineered_flags = ['era_pre_2012',
                     'mths_since_last_delinq_missing', 'mths_since_last_record_missing',
                     'mths_since_recent_bc_dlq_missing', 'mths_since_recent_revol_delinq_missing',
                     'mths_since_last_major_derog_missing', 'emp_length_missing',
                     'mths_since_recent_inq_missing', 'num_tl_120dpd_2m_missing', 'sparse_bureau_missing']
assert len(engineered_flags) == 10

new_features = ['installment_to_income', 'loan_to_income', 'credit_history_months',
                 'revol_bal_to_income', 'open_acc_ratio']
assert len(new_features) == 5

redundant_cols = {'fico_range_high': 'redundancia (r=1.0 com fico_range_low)'}
FEATURE_SET = [c for c in family_C_features if c not in redundant_cols] + engineered_flags + new_features
assert len(FEATURE_SET) == 79

CATEGORICAL_COLS = ['home_ownership', 'purpose', 'verification_status', 'initial_list_status', 'application_type']
REFERENCE_DATE = pd.Timestamp('2000-01-01')


def compute_financials(df):
    interest = (df['installment'] * df['term']) - df['loan_amnt']
    loss_raw = df['loan_amnt'] - df['total_rec_prncp']
    return interest, loss_raw.clip(lower=0)


def prepare_X(df, feature_cols, categorical_cols):
    X = df[feature_cols].copy()
    for c in ['issue_d', 'earliest_cr_line']:
        if c in X.columns:
            X[c] = (X[c] - REFERENCE_DATE).dt.days
    cat_present = [c for c in categorical_cols if c in X.columns]
    X = pd.get_dummies(X, columns=cat_present, drop_first=True)
    return X


def profit_at_threshold(y_true, y_prob, threshold, interest, loss):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    interest = np.asarray(interest)
    loss = np.asarray(loss)
    aprovados = y_prob < threshold
    return interest[aprovados & (y_true == 0)].sum() - loss[aprovados & (y_true == 1)].sum()


def decompose(y_prob, threshold, y_true, interest, loss, loan_amnt):
    y_prob = np.asarray(y_prob)
    y_true = np.asarray(y_true)
    interest = np.asarray(interest)
    loss = np.asarray(loss)
    loan_amnt = np.asarray(loan_amnt)
    rejected = y_prob >= threshold
    rejected_co = rejected & (y_true == 1)
    rejected_fp = rejected & (y_true == 0)
    avoided_loss = loss[rejected_co].sum()
    lost_interest = interest[rejected_fp].sum()
    return {'threshold': threshold, 'n_rejeitados': int(rejected.sum()),
            'valor_total_rejeitado': loan_amnt[rejected].sum(),
            'n_rejeitados_charged_off': int(rejected_co.sum()), 'n_rejeitados_fully_paid': int(rejected_fp.sum()),
            'perda_evitada_$': avoided_loss, 'juros_perdidos_$': lost_interest,
            'ganho_liquido_$': avoided_loss - lost_interest}


threshold_grid = np.round(np.arange(0.01, 1.0, 0.01), 2)

THRESH_M1 = 0.38
THRESH_XGB = 0.31
XGB_WF_PARAMS = dict(max_depth=8, learning_rate=0.03, n_estimators=600, min_child_weight=10,
                     subsample=0.8, colsample_bytree=0.6)

print('Infraestrutura recriada. Thresholds congelados: M1 =', THRESH_M1, '| XGB_walkforward =', THRESH_XGB)


Infraestrutura recriada. Thresholds congelados: M1 = 0.38 | XGB_walkforward = 0.31


## Regra de preprocessing - confirmacao antes de prever

Tudo que foi ajustado no treino e aplicado ao teste sem reajuste. Antes de gerar qualquer
previsao, confirma-se que as colunas de `X_test` sao IDENTICAS as de `X_train` - mesmo
conjunto, mesma ordem. Se nao forem, o notebook para aqui.

In [3]:
X_train = prepare_X(train, FEATURE_SET, CATEGORICAL_COLS)
X_test = prepare_X(test, FEATURE_SET, CATEGORICAL_COLS)

cols_train = X_train.columns.tolist()
cols_test = X_test.columns.tolist()

if cols_train != cols_test:
    only_train = set(cols_train) - set(cols_test)
    only_test = set(cols_test) - set(cols_train)
    print('DIVERGENCIA DE COLUNAS ENTRE TREINO E TESTE.')
    print('Colunas so no treino:', only_train)
    print('Colunas so no teste:', only_test)
    print('Ordem identica (se mesmo conjunto)?', cols_train == cols_test)
    raise RuntimeError('Colunas de treino e teste divergem - preprocessamento parado por seguranca.')
else:
    print(f'OK: {len(cols_train)} colunas identicas, mesma ordem, entre treino e teste.')


OK: 90 colunas identicas, mesma ordem, entre treino e teste.


## Secao 0 - Sanidade do teste (sem olhar performance ainda)

In [4]:
y_test = test['target'].values
n_test = len(test)
default_rate_test = test['target'].mean()

print(f'N do teste: {n_test}')
print(f'Taxa de default no teste: {default_rate_test * 100:.4f}%')
print(f'Intervalo de issue_d no teste: {test["issue_d"].min()} a {test["issue_d"].max()}')
print(f'Todo o teste e 2015? {(test["issue_d"].dt.year == 2015).all()}')


N do teste: 282787
Taxa de default no teste: 14.8836%
Intervalo de issue_d no teste: 2015-01-01 00:00:00 a 2015-12-01 00:00:00
Todo o teste e 2015? True


**Sobre a checagem de interseccao de indices:** os arquivos parquet foram salvos com
indice reiniciado (0..N-1) de forma independente em cada split - o indice bruto do
DataFrame NAO e um identificador global (indice 0 do teste nao tem relacao com indice 0
do treino). A checagem de vazamento equivalente e por CONTEUDO da linha: nenhuma linha do
teste deve ser identica, coluna a coluna, a alguma linha do treino.

In [5]:
train_hashes = set(pd.util.hash_pandas_object(train, index=False))
test_hashes = set(pd.util.hash_pandas_object(test, index=False))
overlap = train_hashes & test_hashes

print(f'Linhas do teste com conteudo identico a alguma linha do treino: {len(overlap)}')
assert len(overlap) == 0, 'ATENCAO: ha linhas duplicadas entre treino e teste.'
print('OK: nenhuma linha do teste e identica a uma linha do treino.')


Linhas do teste com conteudo identico a alguma linha do treino: 0
OK: nenhuma linha do teste e identica a uma linha do treino.


**SMD (differenca padronizada de medias) entre treino e teste, por feature - as 10
maiores.** Contextualiza o quanto a populacao de 2015 difere da populacao de treino
(2007-2013); um certo grau de shift ja era esperado (visto na validacao 2014).

In [6]:
def smd(a, b):
    mean_diff = a.mean() - b.mean()
    pooled_std = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
    return mean_diff / pooled_std if pooled_std > 0 else 0.0

smd_rows = []
for col in X_train.columns:
    smd_rows.append({'feature': col, 'smd': smd(X_train[col], X_test[col])})
smd_df = pd.DataFrame(smd_rows)
smd_df['abs_smd'] = smd_df['smd'].abs()
smd_df = smd_df.sort_values('abs_smd', ascending=False).drop(columns='abs_smd').reset_index(drop=True)
smd_df.head(10)


,feature,smd
0,issue_d,-3.251076
1,initial_list_status_w,-0.927681
2,era_pre_2012,0.922628
3,num_tl_30dpd,-0.916133
4,pct_tl_nvr_dlq,-0.844014
5,mo_sin_old_il_acct,0.808631
6,num_tl_120dpd_2m_missing,0.695062
7,num_tl_120dpd_2m,-0.694251
8,percent_bc_gt_75,0.689111
9,bc_util,0.687600


## Secao 1 - Treinar os dois finalistas no treino (configuracao congelada)

Hiperparametros e thresholds sao exatamente os fixados no notebook 11. Nenhum ajuste
ocorre neste notebook. O treino de XGBoost usa a ordem original das linhas de
`train.parquet` (sem reordenar por `issue_d`), consistente com a correcao de causa
registrada no notebook 11.

In [7]:
y_train = train['target'].values

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    m1_model = LogisticRegression(penalty='l2', solver='lbfgs', max_iter=2000, random_state=42)
    m1_model.fit(X_train_scaled, y_train)
    convergence_warnings = [x for x in w if 'onverg' in str(x.message)]

n_iter = m1_model.n_iter_[0]
print(f'M1: n_iter_={n_iter} (max_iter=2000) | ConvergenceWarning emitido? {len(convergence_warnings) > 0}')
if n_iter >= 2000 or len(convergence_warnings) > 0:
    raise RuntimeError('M1 nao convergiu dentro de max_iter=2000.')
print('M1 convergiu.')

xgb_wf = XGBClassifier(**XGB_WF_PARAMS, random_state=42, eval_metric='logloss', n_jobs=1)
xgb_wf.fit(X_train, y_train)
print('XGB_walkforward treinado.')


M1: n_iter_=84 (max_iter=2000) | ConvergenceWarning emitido? False
M1 convergiu.


XGB_walkforward treinado.


In [8]:
# Previsoes no teste (avaliacao) e na validacao (apenas para recalcular os numeros ja
# fixados no notebook 11, para a comparacao da Secao 4 - nenhum ajuste e feito com base nisso)
X_val = prepare_X(validation, FEATURE_SET, CATEGORICAL_COLS)
X_val = X_val.reindex(columns=X_train.columns, fill_value=0)
X_val_scaled = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns, index=X_val.index)
y_val = validation['target'].values

interest_test, loss_test = compute_financials(test)
interest_test = interest_test.values
loss_test = loss_test.values
loan_amnt_test = test['loan_amnt'].values

interest_val, loss_val = compute_financials(validation)
interest_val = interest_val.values
loss_val = loss_val.values

y_prob_test_m1 = m1_model.predict_proba(X_test_scaled)[:, 1]
y_prob_test_xgb = xgb_wf.predict_proba(X_test)[:, 1]

y_prob_val_m1 = m1_model.predict_proba(X_val_scaled)[:, 1]
y_prob_val_xgb = xgb_wf.predict_proba(X_val)[:, 1]

print('Previsoes geradas no teste e na validacao (validacao so para referencia, Secao 4).')


Previsoes geradas no teste e na validacao (validacao so para referencia, Secao 4).


### Bootstrap no teste (1.000 reamostras, seed=42) - alimenta as Secoes 2 e 3

Um unico bootstrap sobre o teste, guardando por reamostra: AUC-ROC, AUC-PR e Brier de
M1/XGB, e o lucro de M0b/M1/XGB no threshold congelado. As Secoes 2 e 3 sao lidas a partir
deste mesmo conjunto de reamostras.

In [9]:
N_BOOT = 1000
SEED = 42
rng = np.random.default_rng(SEED)

boot = {
    'auc_m1': np.zeros(N_BOOT), 'auc_xgb': np.zeros(N_BOOT),
    'auc_pr_m1': np.zeros(N_BOOT), 'auc_pr_xgb': np.zeros(N_BOOT),
    'brier_m1': np.zeros(N_BOOT), 'brier_xgb': np.zeros(N_BOOT),
    'lucro_m0b': np.zeros(N_BOOT), 'lucro_m1': np.zeros(N_BOOT), 'lucro_xgb': np.zeros(N_BOOT),
}

for b in range(N_BOOT):
    idx = rng.integers(0, n_test, size=n_test)
    yb = y_test[idx]
    ib = interest_test[idx]
    lb = loss_test[idx]
    pm1 = y_prob_test_m1[idx]
    pxgb = y_prob_test_xgb[idx]

    boot['auc_m1'][b] = roc_auc_score(yb, pm1)
    boot['auc_xgb'][b] = roc_auc_score(yb, pxgb)
    boot['auc_pr_m1'][b] = average_precision_score(yb, pm1)
    boot['auc_pr_xgb'][b] = average_precision_score(yb, pxgb)
    boot['brier_m1'][b] = brier_score_loss(yb, pm1)
    boot['brier_xgb'][b] = brier_score_loss(yb, pxgb)
    boot['lucro_m0b'][b] = ib[yb == 0].sum() - lb[yb == 1].sum()
    boot['lucro_m1'][b] = profit_at_threshold(yb, pm1, THRESH_M1, ib, lb)
    boot['lucro_xgb'][b] = profit_at_threshold(yb, pxgb, THRESH_XGB, ib, lb)

print(f'Bootstrap concluido. N_BOOT={N_BOOT} seed={SEED}')


Bootstrap concluido. N_BOOT=1000 seed=42


## Secao 2 - Resultado no teste

Ponto estimado + IC 95% (percentil, das 1.000 reamostras acima) - nao valores pontuais de
precisao falsa.

In [10]:
def ci(arr):
    lo, hi = np.percentile(arr, [2.5, 97.5])
    return lo, hi

lucro_m0b_test = interest_test[y_test == 0].sum() - loss_test[y_test == 1].sum()
aprov_m1_test = y_prob_test_m1 < THRESH_M1
aprov_xgb_test = y_prob_test_xgb < THRESH_XGB

lo, hi = ci(boot['lucro_m0b'])
print(f'M0b (aprova todos): lucro = $ {lucro_m0b_test:,.2f} | IC95% [$ {lo:,.2f}, $ {hi:,.2f}]')
print(f'  % aprovados: 100.00% | default entre aprovados: {default_rate_test * 100:.4f}%')
print()

auc_m1 = roc_auc_score(y_test, y_prob_test_m1)
auc_pr_m1 = average_precision_score(y_test, y_prob_test_m1)
brier_m1 = brier_score_loss(y_test, y_prob_test_m1)
lucro_m1_test = profit_at_threshold(y_test, y_prob_test_m1, THRESH_M1, interest_test, loss_test)
lo_auc, hi_auc = ci(boot['auc_m1'])
lo_brier, hi_brier = ci(boot['brier_m1'])
lo_lucro, hi_lucro = ci(boot['lucro_m1'])
print(f'M1 (threshold={THRESH_M1}):')
print(f'  AUC-ROC = {auc_m1:.4f} | IC95% [{lo_auc:.4f}, {hi_auc:.4f}]')
print(f'  AUC-PR = {auc_pr_m1:.4f}')
print(f'  Brier = {brier_m1:.4f} | IC95% [{lo_brier:.4f}, {hi_brier:.4f}]')
print(f'  Lucro = $ {lucro_m1_test:,.2f} | IC95% [$ {lo_lucro:,.2f}, $ {hi_lucro:,.2f}]')
print(f'  % aprovados = {aprov_m1_test.mean() * 100:.2f}% | default entre aprovados = {y_test[aprov_m1_test].mean() * 100:.4f}%')
print()

auc_xgb = roc_auc_score(y_test, y_prob_test_xgb)
auc_pr_xgb = average_precision_score(y_test, y_prob_test_xgb)
brier_xgb = brier_score_loss(y_test, y_prob_test_xgb)
lucro_xgb_test = profit_at_threshold(y_test, y_prob_test_xgb, THRESH_XGB, interest_test, loss_test)
lo_auc_x, hi_auc_x = ci(boot['auc_xgb'])
lo_brier_x, hi_brier_x = ci(boot['brier_xgb'])
lo_lucro_x, hi_lucro_x = ci(boot['lucro_xgb'])
print(f'XGB_walkforward (threshold={THRESH_XGB}):')
print(f'  AUC-ROC = {auc_xgb:.4f} | IC95% [{lo_auc_x:.4f}, {hi_auc_x:.4f}]')
print(f'  AUC-PR = {auc_pr_xgb:.4f}')
print(f'  Brier = {brier_xgb:.4f} | IC95% [{lo_brier_x:.4f}, {hi_brier_x:.4f}]')
print(f'  Lucro = $ {lucro_xgb_test:,.2f} | IC95% [$ {lo_lucro_x:,.2f}, $ {hi_lucro_x:,.2f}]')
print(f'  % aprovados = {aprov_xgb_test.mean() * 100:.2f}% | default entre aprovados = {y_test[aprov_xgb_test].mean() * 100:.4f}%')


M0b (aprova todos): lucro = $ 233,202,813.06 | IC95% [$ 228,626,556.25, $ 237,526,093.71]
  % aprovados: 100.00% | default entre aprovados: 14.8836%

M1 (threshold=0.38):
  AUC-ROC = 0.6847 | IC95% [0.6820, 0.6874]
  AUC-PR = 0.2617
  Brier = 0.1210 | IC95% [0.1200, 0.1220]
  Lucro = $ 235,936,408.63 | IC95% [$ 231,368,999.00, $ 240,458,423.72]
  % aprovados = 98.88% | default entre aprovados = 14.6184%



XGB_walkforward (threshold=0.31):
  AUC-ROC = 0.6846 | IC95% [0.6818, 0.6871]
  AUC-PR = 0.2636
  Brier = 0.1205 | IC95% [0.1195, 0.1214]
  Lucro = $ 242,230,710.89 | IC95% [$ 237,888,392.29, $ 246,720,384.43]
  % aprovados = 96.24% | default entre aprovados = 14.0492%


In [11]:
resultado_teste_df = pd.DataFrame([
    {'modelo': 'M0b', 'auc_roc': np.nan, 'brier': np.nan, 'threshold': np.nan,
     'lucro': lucro_m0b_test, 'lucro_ic_low': ci(boot['lucro_m0b'])[0], 'lucro_ic_high': ci(boot['lucro_m0b'])[1],
     'pct_aprovados': 100.0, 'default_aprovados': default_rate_test * 100},
    {'modelo': 'M1', 'auc_roc': auc_m1, 'brier': brier_m1, 'threshold': THRESH_M1,
     'lucro': lucro_m1_test, 'lucro_ic_low': lo_lucro, 'lucro_ic_high': hi_lucro,
     'pct_aprovados': aprov_m1_test.mean() * 100, 'default_aprovados': y_test[aprov_m1_test].mean() * 100},
    {'modelo': 'XGB_walkforward', 'auc_roc': auc_xgb, 'brier': brier_xgb, 'threshold': THRESH_XGB,
     'lucro': lucro_xgb_test, 'lucro_ic_low': lo_lucro_x, 'lucro_ic_high': hi_lucro_x,
     'pct_aprovados': aprov_xgb_test.mean() * 100, 'default_aprovados': y_test[aprov_xgb_test].mean() * 100},
]).set_index('modelo')
resultado_teste_df


,auc_roc,brier,threshold,lucro,lucro_ic_low,lucro_ic_high,pct_aprovados,default_aprovados
modelo,,,,,,,,
M0b,NaN,NaN,NaN,2.332028e+08,2.286266e+08,2.375261e+08,100.000000,14.883640
M1,0.684718,0.121035,0.38,2.359364e+08,2.313690e+08,2.404584e+08,98.877954,14.618367
XGB_walkforward,0.684563,0.120469,0.31,2.422307e+08,2.378884e+08,2.467204e+08,96.236036,14.049231


## Secao 3 - Bootstrap no teste: diferencas

In [12]:
diff_m1_m0b = boot['lucro_m1'] - boot['lucro_m0b']
diff_xgb_m0b = boot['lucro_xgb'] - boot['lucro_m0b']
diff_m1_xgb = boot['lucro_m1'] - boot['lucro_xgb']

for name, d in [('M1 vs M0b', diff_m1_m0b), ('XGB_walkforward vs M0b', diff_xgb_m0b)]:
    lo, hi = ci(d)
    print(f'{name}: media = $ {d.mean():,.2f} | IC95% [$ {lo:,.2f}, $ {hi:,.2f}] | % positivas = {(d > 0).mean() * 100:.2f}%')

print()
lo, hi = ci(diff_m1_xgb)
cruza_zero = lo < 0 < hi
print(f'M1 vs XGB_walkforward (pareado): media = $ {diff_m1_xgb.mean():,.2f} | IC95% [$ {lo:,.2f}, $ {hi:,.2f}]')
print(f'  Cruza zero? {cruza_zero}')
print(f'  % de reamostras em que M1 venceu: {(diff_m1_xgb > 0).mean() * 100:.2f}%')
print(f'  % de reamostras em que XGB_walkforward venceu: {(diff_m1_xgb < 0).mean() * 100:.2f}%')


M1 vs M0b: media = $ 2,750,813.30 | IC95% [$ 2,037,165.24, $ 3,542,444.80] | % positivas = 100.00%
XGB_walkforward vs M0b: media = $ 9,068,587.76 | IC95% [$ 7,627,319.06, $ 10,664,983.20] | % positivas = 100.00%

M1 vs XGB_walkforward (pareado): media = $ -6,317,774.45 | IC95% [$ -7,706,571.79, $ -4,931,521.12]
  Cruza zero? False
  % de reamostras em que M1 venceu: 0.00%
  % de reamostras em que XGB_walkforward venceu: 100.00%


**Leitura obrigatoria:** se o IC pareado M1 vs XGB_walkforward cruzar zero, os dois
modelos sao estatisticamente indistinguiveis em lucro no teste - nao se declara vencedor
por lucro, e a decisao entre eles passa a depender de explicabilidade/robustez, nao de
uma diferenca de performance que os dados sustentem.

## Secao 4 - Validacao -> Teste: quanto o ganho encolheu

Ganho de cada modelo sobre M0b, comparando o numero ja conhecido da validacao (recalculado
aqui com o MESMO modelo congelado, so para garantir consistencia - nao houve reajuste) com
o numero do teste.

In [13]:
lucro_m0b_val = interest_val[y_val == 0].sum() - loss_val[y_val == 1].sum()
ganho_val_m1 = profit_at_threshold(y_val, y_prob_val_m1, THRESH_M1, interest_val, loss_val) - lucro_m0b_val
ganho_val_xgb = profit_at_threshold(y_val, y_prob_val_xgb, THRESH_XGB, interest_val, loss_val) - lucro_m0b_val

ganho_test_m1 = lucro_m1_test - lucro_m0b_test
ganho_test_xgb = lucro_xgb_test - lucro_m0b_test

shrink_rows = []
for name, gv, gt in [('M1', ganho_val_m1, ganho_test_m1), ('XGB_walkforward', ganho_val_xgb, ganho_test_xgb)]:
    encolhimento = gv - gt
    pct = (encolhimento / gv * 100) if gv != 0 else np.nan
    shrink_rows.append({'modelo': name, 'ganho_validacao_$': gv, 'ganho_teste_$': gt,
                         'encolhimento_$': encolhimento, 'encolhimento_%': pct})
shrink_df = pd.DataFrame(shrink_rows).set_index('modelo')
shrink_df


,ganho_validacao_$,ganho_teste_$,encolhimento_$,encolhimento_%
modelo,,,,
M1,874222.01,2733595.57,-1859373.56,-212.688944
XGB_walkforward,1865541.47,9027897.83,-7162356.36,-383.929088


## Secao 5 - Decomposicao do resultado no teste (modelo de maior lucro)

In [14]:
melhor_modelo = 'M1' if lucro_m1_test >= lucro_xgb_test else 'XGB_walkforward'
print(f'Modelo com maior lucro no teste: {melhor_modelo}')

if melhor_modelo == 'M1':
    y_prob_best = y_prob_test_m1
    thresh_best = THRESH_M1
    lucro_best = lucro_m1_test
else:
    y_prob_best = y_prob_test_xgb
    thresh_best = THRESH_XGB
    lucro_best = lucro_xgb_test

decomp_result = decompose(y_prob_best, thresh_best, y_test, interest_test, loss_test, loan_amnt_test)
for k, v in decomp_result.items():
    if isinstance(v, float):
        print(f'{k}: $ {v:,.2f}' if 'ganho' in k or 'perda' in k or 'juros' in k or 'valor' in k else f'{k}: {v}')
    else:
        print(f'{k}: {v}')

ganho_liquido_check = decomp_result['ganho_liquido_$']
ganho_esperado = lucro_best - lucro_m0b_test
print()
print(f'Ganho liquido (decomposicao): $ {ganho_liquido_check:,.2f}')
print(f'Lucro do modelo - Lucro M0b (Secao 2): $ {ganho_esperado:,.2f}')
print(f'Batem (diferenca < $1)? {abs(ganho_liquido_check - ganho_esperado) < 1}')


Modelo com maior lucro no teste: XGB_walkforward
threshold: 0.31
n_rejeitados: 10644
valor_total_rejeitado: $ 136,302,375.00
n_rejeitados_charged_off: 3855
n_rejeitados_fully_paid: 6789
perda_evitada_$: $ 32,151,026.71
juros_perdidos_$: $ 23,123,128.88
ganho_liquido_$: $ 9,027,897.83

Ganho liquido (decomposicao): $ 9,027,897.83
Lucro do modelo - Lucro M0b (Secao 2): $ 9,027,897.83
Batem (diferenca < $1)? True


## Secao 6 - Analise de transferencia (60 meses)

Ambos os modelos foram treinados nos 36 meses (train.parquet). Aqui sao aplicados, sem
qualquer reajuste, ao `transfer_60m.parquet`.

In [15]:
X_transfer = prepare_X(transfer_60m, FEATURE_SET, CATEGORICAL_COLS)
cols_transfer = X_transfer.columns.tolist()
if cols_transfer != cols_train:
    only_train_t = set(cols_train) - set(cols_transfer)
    only_transfer = set(cols_transfer) - set(cols_train)
    print('Aviso: colunas de transfer_60m divergem do treino.')
    print('So no treino:', only_train_t, '| So em transfer_60m:', only_transfer)
    X_transfer = X_transfer.reindex(columns=X_train.columns, fill_value=0)
else:
    print('OK: colunas de transfer_60m identicas as do treino.')

X_transfer_scaled = pd.DataFrame(scaler.transform(X_transfer), columns=X_transfer.columns, index=X_transfer.index)
y_transfer = transfer_60m['target'].values
interest_transfer, loss_transfer = compute_financials(transfer_60m)
interest_transfer = interest_transfer.values
loss_transfer = loss_transfer.values

default_rate_transfer = transfer_60m['target'].mean()
print(f'N transfer_60m: {len(transfer_60m)} | taxa de default: {default_rate_transfer * 100:.2f}%')
print(f'Taxa de default no teste (36m): {default_rate_test * 100:.2f}%')


OK: colunas de transfer_60m identicas as do treino.


N transfer_60m: 54969 | taxa de default: 25.16%
Taxa de default no teste (36m): 14.88%


In [16]:
y_prob_transfer_m1 = m1_model.predict_proba(X_transfer_scaled)[:, 1]
y_prob_transfer_xgb = xgb_wf.predict_proba(X_transfer)[:, 1]

lucro_m0b_transfer = interest_transfer[y_transfer == 0].sum() - loss_transfer[y_transfer == 1].sum()

auc_transfer_m1 = roc_auc_score(y_transfer, y_prob_transfer_m1)
lucro_transfer_m1 = profit_at_threshold(y_transfer, y_prob_transfer_m1, THRESH_M1, interest_transfer, loss_transfer)
ganho_transfer_m1 = lucro_transfer_m1 - lucro_m0b_transfer

auc_transfer_xgb = roc_auc_score(y_transfer, y_prob_transfer_xgb)
lucro_transfer_xgb = profit_at_threshold(y_transfer, y_prob_transfer_xgb, THRESH_XGB, interest_transfer, loss_transfer)
ganho_transfer_xgb = lucro_transfer_xgb - lucro_m0b_transfer

transfer_rows = []
for name, auc_36, auc_60, ganho_36, ganho_60 in [
    ('M1', auc_m1, auc_transfer_m1, ganho_test_m1, ganho_transfer_m1),
    ('XGB_walkforward', auc_xgb, auc_transfer_xgb, ganho_test_xgb, ganho_transfer_xgb),
]:
    transfer_rows.append({
        'modelo': name, 'auc_roc_36m': auc_36, 'auc_roc_60m': auc_60, 'delta_auc': auc_60 - auc_36,
        'ganho_vs_m0b_36m_$': ganho_36, 'ganho_vs_m0b_60m_$': ganho_60,
        'delta_ganho_%': (ganho_60 - ganho_36) / ganho_36 * 100 if ganho_36 != 0 else np.nan,
    })
transfer_df = pd.DataFrame(transfer_rows).set_index('modelo')
transfer_df


,auc_roc_36m,auc_roc_60m,delta_auc,ganho_vs_m0b_36m_$,ganho_vs_m0b_60m_$,delta_ganho_%
modelo,,,,,,
M1,0.684718,0.61198,-0.072738,2733595.57,-20472.60,-100.748926
XGB_walkforward,0.684563,0.64332,-0.041243,9027897.83,1066477.36,-88.186869


**Leitura:** degradacao leve de AUC/ganho entre 36m e 60m sugere que os dois prazos
compartilham o mesmo comportamento de risco subjacente (um scorecard unico bastaria).
Degradacao severa sugere populacoes estruturalmente distintas (achado sobre o dataset,
nao falha do modelo) - a taxa de default base ja e visivelmente mais alta em 60m do que
em 36m (ver celula acima), o que por si so aponta para populacoes com composicao de risco
diferente.

## Secao 7 - Nota de leitura

**Este e o resultado final do projeto. Nao ha reajuste apos esta nota.**

**(a) Qual modelo tem maior lucro no teste:** XGB_walkforward, com $ 242.230.710,89
contra $ 235.936.408,63 de M1 - uma vantagem de $ 6.317.774,45 no threshold ja
congelado de cada um (0,31 e 0,38, respectivamente). As duas AUC-ROC no teste sao
praticamente identicas (0,6846 vs 0,6847) - a vantagem de XGB_walkforward esta quase
inteiramente na forma como o lucro se acumula perto do ponto de corte, nao na
capacidade de ordenar risco de forma geral.

**(b) A diferenca e estatisticamente distinguivel:** SIM, e de forma clara. O IC 95%
pareado (M1 - XGB_walkforward) e [-$ 7.706.571,79, -$ 4.931.521,12] - nao cruza zero,
e XGB_walkforward venceu em 100% das 1.000 reamostras. Isso contrasta com a leitura mais
cautelosa que foi necessaria na validacao (notebook 10/11, onde comparacoes entre
configuracoes de XGBoost tinham margens mais estreitas e sujeitas a ruido de
reexecucao/ordem dos dados). Aqui, com a populacao de teste maior (282.787 linhas) e uma
vantagem de lucro de quase 3% do lucro total, a distincao e robusta - **XGB_walkforward
e declarado vencedor por lucro nesta avaliacao.**

**(c) Quanto o ganho encolheu, validacao -> teste:** NAO encolheu - cresceu, e de forma
acentuada, nos dois modelos:
- M1: de $ 874.222,01 (validacao) para $ 2.733.595,57 (teste) - crescimento de +213%.
- XGB_walkforward: de $ 1.865.541,47 (validacao) para $ 9.027.897,83 (teste) -
  crescimento de +384%.

Isto e o OPOSTO do que "overfitting do processo de selecao" preveria (a expectativa
usual e que o ganho medido na validacao encolha no teste, nao cresca). Parte do
crescimento e mecanica - o teste tem ~74% mais linhas que a validacao (282.787 vs
162.570), entao o ganho absoluto em dolares tende a escalar com o tamanho da carteira.
Mas mesmo em termos RELATIVOS (ganho / lucro de M0b), a vantagem cresceu: XGB_walkforward
passou de 0,98% do lucro de M0b na validacao para 3,87% no teste. Isso e uma noticia
favoravel - o processo de seguir a validacao ate aqui (walk-forward, calibracao,
quantificacao de ruido de reexecucao) nao parece ter produzido um modelo que so
funcionava na validacao - mas e um UNICO teste, sem repeticao: nao ha como saber, so com
este resultado, se o crescimento do ganho e um padrao que se repetiria em novas safras ou
uma particularidade favoravel da safra de 2015. Nao se deve extrapolar que o ganho vai
continuar crescendo.

**(d) O que a transferencia para 60 meses revelou:** degradacao SEVERA nos dois modelos,
nao leve. A taxa de default base quase dobra (25,16% em 60m contra 14,88% em 36m no
teste). A AUC-ROC cai em ambos (M1: 0,6847 -> 0,6120, -7,3 pontos percentuais;
XGB_walkforward: 0,6846 -> 0,6433, -4,1 pontos percentuais), e o ganho sobre M0b quase
desaparece ou fica negativo:
- M1: de +$ 2.733.595,57 (36m) para **-$ 20.472,60 (60m)** - pior que aprovar todo mundo.
- XGB_walkforward: de +$ 9.027.897,83 (36m) para +$ 1.066.477,36 (60m) - perda de 88% do
  ganho, mas ainda positivo.

Pela leitura combinada no enunciado deste notebook, esta e evidencia de populacoes
estruturalmente distintas entre 36 e 60 meses - nao o mesmo comportamento de risco. Um
scorecard treinado em emprestimos de 36 meses nao deveria ser usado, sem recalibracao ou
um modelo dedicado, para decidir sobre emprestimos de 60 meses: isso e um achado sobre a
estrutura do produto/dataset, nao uma falha dos modelos. Vale notar que XGB_walkforward
degrada MENOS que M1 nos dois eixos (AUC e lucro) - sua fronteira de decisao generaliza
um pouco melhor para a populacao de maior risco, embora ainda de forma severa.

**(e) Vies de selecao do dataset - leitura obrigatoria:** todo numero de lucro reportado
neste notebook, do inicio ao fim do projeto, mede desempenho sobre a populacao de
emprestimos que a LendingClub JA APROVOU e originou. O modelo nunca viu, e nao pode ser
avaliado sobre, os solicitantes que a LC rejeitou antes mesmo de conceder o emprestimo.
Isso significa que os numeros de lucro aqui estimam quanto um SEGUNDO FILTRO, aplicado
sobre a carteira que a LC ja decidiu aprovar, agregaria de valor - NAO uma estimativa de
quanto lucro um sistema de concessao de credito construido do zero, decidindo sobre TODOS
os solicitantes (aprovados e rejeitados pela LC), geraria. Extrapolar estes numeros para
uma politica de credito que decida sobre a populacao geral de solicitantes superestimaria
o resultado - a populacao real de solicitantes inclui, presumivelmente, casos de risco
mais alto que a LC ja filtrou e que nenhum dos modelos aqui jamais viu.

**Resultado final, sem reajuste:** XGB_walkforward (max_depth=8, learning_rate=0.03,
n_estimators=600, min_child_weight=10, subsample=0.8, colsample_bytree=0.6, threshold
0,31) e o modelo de maior lucro no teste de 2015, com vantagem estatisticamente
distinguivel sobre M1. A transferencia para 60 meses mostra que este resultado NAO se
generaliza para emprestimos de prazo mais longo sem trabalho adicional. O lucro medido
reflete apenas a populacao ja aprovada pela LendingClub.